# CosMx → Celldega pre-processing (standalone)

Generates Celldega **DegaFiles** (LandscapeFiles) for a CosMx SMI dataset by **converting CosMx flat files into the Xenium file format** and then reusing the existing `celldega.pre` Xenium pre-processing functions (no changes to the `pre` module).

**Dataset:** colon CRC discovery sample `S0` (NanoString WTx, 18,935 genes, ~494k cells, ~1.12B transcripts). Morphology images come from the OME-ZARR inside `Napari.zip`.

### Key facts / decisions
* **Coordinate frame:** CosMx `*_global_px` == Napari image level-0 pixels (verified: flat-file px span 114925×120960 ≈ image 114908×120942). So cells/transcripts need **no** coordinate transform to align with the morphology image.
* **Polygons fix:** CosMx flat-file polygons have correct cell shape/size but wrong global position (per-FOV centroids spread 2×). We keep the shape and translate each polygon so its centroid matches the (validated) metadata centroid.
* **Image scale:** images are built at zarr **level 2** (0.48 µm/px, 28727×30235). We map `global_px → level-2 px` by baking a `0.25` scale into the transform matrix (NOT via `image_scale`, which celldega applies inconsistently across functions, so keep it = 1).
* **Image flip:** the Napari OME-ZARR stores the mosaic vertically flipped relative to the CosMx `global_px` frame (cell-density-by-y correlates +0.84 with flipped image rows), so the image is `np.flipud`-ed to align with cells/transcripts.
* **Transcripts (ALL of them):** celldega's `make_trx_tiles` loads the whole transcript parquet into RAM (1.12B rows → OOM). We stream the input in pyarrow batches and spill/merge per tile, so **no subsampling** is needed. (A sequential `id % N` subsample would also stripe, since the tx file is sorted by x.)
* **Non-row-group layout:** classic one-file-per-tile/gene. Row-group DegaFiles have a known 403 issue when hosted on the Manifold S3 bucket (chunked-parquet Range requests).
* Run everything on fast local disk; sync DegaFiles to the bucket with `aws s3 sync`.

**Requirements:** `celldega` (v0.18.0), `pyvips` (with system libvips), `pyarrow`, `polars`, `scanpy`, `scipy`, `zarr`.

Edit the paths/constants at the top of each step for your environment.

## Step 1 — metadata + polygons → Xenium `cells.csv.gz` and `cell_boundaries.parquet`
Run the final `gzip` to produce `cells.csv.gz` after this cell (the cell writes `cells.csv`).

In [ ]:
"""Convert CosMx metadata + polygons -> Xenium-format cells.csv.gz and cell_boundaries.parquet.

CosMx flat-file polygons have the CORRECT cell shape/size (raw bbox == metadata Width/Height,
ratio 1.00) but their global POSITION is wrong: within each FOV the per-cell centroids are spread
~2x relative to the true (metadata/transcript/image) frame. So we keep the polygon shape and
translate each cell's polygon so its centroid lands on the validated metadata centroid (which is
confirmed to align with the morphology image):
    vertex_fixed = raw_vertex - mean(raw_vertices_of_cell) + metadata_centroid_of_cell
"""
import polars as pl
from pathlib import Path

COSMX = Path("/home/jovyan/local/cosmx_data")
OUT = Path("/home/jovyan/local/cosmx_xenium/S0")
OUT.mkdir(parents=True, exist_ok=True)

# ---- cells.csv.gz (Xenium: cell_id, x_centroid, y_centroid) ----
meta = pl.read_csv(
    COSMX / "S0_metadata_file.csv.gz",
    columns=["cell", "CenterX_global_px", "CenterY_global_px"],
    schema_overrides={"CenterX_global_px": pl.Float64, "CenterY_global_px": pl.Float64},
)
cells = meta.rename(
    {"cell": "cell_id", "CenterX_global_px": "x_centroid", "CenterY_global_px": "y_centroid"}
).select(["cell_id", "x_centroid", "y_centroid"])
print("cells:", cells.shape)
cells.write_csv(OUT / "cells.csv")  # gzip after with shell

# ---- cell_boundaries.parquet (Xenium: cell_id, vertex_x, vertex_y) ----
# Keep polygon shape, translate so each cell's polygon centroid == metadata centroid.
poly = pl.read_csv(
    COSMX / "S0-polygons.csv.gz",
    columns=["cell", "x_global_px", "y_global_px"],
    schema_overrides={"x_global_px": pl.Float64, "y_global_px": pl.Float64},
).rename({"cell": "cell_id"})
# per-cell raw polygon centroid (mean of vertices)
raw_c = poly.group_by("cell_id").agg([
    pl.col("x_global_px").mean().alias("rcx"),
    pl.col("y_global_px").mean().alias("rcy"),
])
poly = poly.join(raw_c, on="cell_id", how="left").join(
    cells.rename({"x_centroid": "mcx", "y_centroid": "mcy"}), on="cell_id", how="inner"
)
bounds = poly.with_columns([
    (pl.col("x_global_px") - pl.col("rcx") + pl.col("mcx")).alias("vertex_x"),
    (pl.col("y_global_px") - pl.col("rcy") + pl.col("mcy")).alias("vertex_y"),
]).select(["cell_id", "vertex_x", "vertex_y"])
# drop degenerate polygons (<4 vertices) — shapely LinearRing needs >=4 coordinates
_keep = bounds.group_by("cell_id").len().filter(pl.col("len") >= 4).select("cell_id")
bounds = bounds.join(_keep, on="cell_id", how="inner")
print("boundaries rows:", bounds.shape, "unique cells:", bounds["cell_id"].n_unique())
bounds.write_parquet(OUT / "cell_boundaries.parquet")
print("DONE cells+boundaries")


In [ ]:
import subprocess
subprocess.run(['gzip', '-f', '/home/jovyan/local/cosmx_xenium/S0/cells.csv'], check=True)
print('cells.csv.gz written')

## Step 2 — expression matrix → 10x MTX `cell_feature_matrix/`
pyarrow streaming, parses only real genes (drops `Negative*` / `SystemControl*`). ~8 min; the final `mmwrite` of ~550M nonzeros dominates.

In [ ]:
"""Convert CosMx exprMat CSV -> 10x MTX (cell_feature_matrix/) using pyarrow streaming (fast).

Drops control columns (Negative*, SystemControl*). pyarrow parses only the kept columns.
Writes matrix.mtx.gz (genes x cells), barcodes.tsv.gz, features.tsv.gz so read_cbg_mtx works
(read_cbg_mtx transposes -> cells x genes indexed by barcodes, cols = features[1]).
"""
import gzip, shutil, time
from pathlib import Path
import numpy as np
import pyarrow as pa
import pyarrow.csv as pacsv
import scipy.sparse as sp
from scipy.io import mmwrite

COSMX = "/home/jovyan/local/cosmx_data/S0_exprMat_file.csv.gz"
OUTDIR = Path("/home/jovyan/local/cosmx_xenium/S0/cell_feature_matrix")
OUTDIR.mkdir(parents=True, exist_ok=True)
SLIDE = 1

# header
with gzip.open(COSMX, "rt") as f:
    header = f.readline().strip().split(",")
genes = [g for g in header[2:]
         if not (g.startswith("Negative") or g.startswith("SystemControl"))]
print(f"kept genes {len(genes)} of {len(header)-2}")

include = ["fov", "cell_ID"] + genes
col_types = {"fov": pa.int32(), "cell_ID": pa.int64()}
for g in genes:
    col_types[g] = pa.int32()

read_opts = pacsv.ReadOptions(block_size=256 << 20)
conv_opts = pacsv.ConvertOptions(include_columns=include, column_types=col_types)

t = time.time()
reader = pacsv.open_csv(COSMX, read_options=read_opts, convert_options=conv_opts)
barcodes = []
blocks = []
nrows = 0
for batch in reader:
    fov = batch.column("fov").to_numpy()
    cid = batch.column("cell_ID").to_numpy()
    barcodes.extend(f"c_{SLIDE}_{f}_{c}" for f, c in zip(fov, cid))
    # gene matrix: stack the gene columns (zero-copy int32 arrays) -> dense -> csr
    mat = np.empty((batch.num_rows, len(genes)), dtype=np.int32)
    for j, g in enumerate(genes):
        mat[:, j] = batch.column(g).to_numpy(zero_copy_only=False)
    blocks.append(sp.csr_matrix(mat))
    nrows += batch.num_rows
    print(f"  {nrows} cells  {time.time()-t:.0f}s", flush=True)

cbg = sp.vstack(blocks).tocsr()
print("cbg (cells x genes):", cbg.shape, "nnz", cbg.nnz, "%.0fs" % (time.time()-t))

with gzip.open(OUTDIR / "barcodes.tsv.gz", "wt") as f:
    f.write("\n".join(barcodes) + "\n")
with gzip.open(OUTDIR / "features.tsv.gz", "wt") as f:
    f.write("\n".join(f"{g}\t{g}\tGene Expression" for g in genes) + "\n")
mtx = OUTDIR / "matrix.mtx"
mmwrite(str(mtx), cbg.T.tocoo(), field="integer")
with open(mtx, "rb") as fi, gzip.open(str(mtx) + ".gz", "wb") as fo:
    shutil.copyfileobj(fi, fo, length=16 << 20)
mtx.unlink()
print("DONE %.0fs" % (time.time()-t), sorted(p.name for p in OUTDIR.iterdir()))


## Step 3 — transcripts → Xenium `transcripts.parquet` (ALL transcripts)
pyarrow streams the 10.6 GB gzip (bounded memory) and writes large row groups for fast downstream parsing. **No subsampling** — all ~1.12B transcripts are kept; the streaming tiler in Step 4 handles them in bounded memory.

In [ ]:
"""Convert CosMx tx_file (10.6GB gz) -> Xenium-format transcripts.parquet via pyarrow streaming.

Streams the gzipped CSV batch-by-batch (bounded memory), filters control targets, renames to
Xenium columns (cell_id, transcript_id, feature_name, x_location, y_location). Coords are already
in the image frame.
"""
import time
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
import pyarrow.parquet as pq

TX = "/home/jovyan/local/cosmx_data/S0_tx_file.csv.gz"
OUT = "/home/jovyan/local/cosmx_xenium/S0/transcripts.parquet"

read_opts = pacsv.ReadOptions(block_size=256 << 20)  # 256MB blocks -> larger parquet row groups
# (larger row groups parse faster downstream, e.g. in the streaming transcript tiler)
parse_opts = pacsv.ParseOptions(delimiter=",")
conv_opts = pacsv.ConvertOptions(
    column_types={"x_global_px": pa.float64(), "y_global_px": pa.float64(),
                  "fov": pa.int32(), "cell_ID": pa.int64()},
    include_columns=["cell", "target", "x_global_px", "y_global_px"],
)

out_schema = pa.schema([
    ("cell_id", pa.string()),
    ("transcript_id", pa.int64()),
    ("feature_name", pa.string()),
    ("x_location", pa.float64()),
    ("y_location", pa.float64()),
])

t = time.time()
reader = pacsv.open_csv(TX, read_options=read_opts, parse_options=parse_opts, convert_options=conv_opts)
writer = pq.ParquetWriter(OUT, out_schema, compression="zstd")
tid = 0
nrows = 0
nbatch = 0
for batch in reader:
    tgt = batch.column("target")
    is_ctrl = pc.or_(pc.starts_with(tgt, "Negative"), pc.starts_with(tgt, "SystemControl"))
    keep = pc.invert(is_ctrl)
    b = batch.filter(keep)
    n = b.num_rows
    if n == 0:
        continue
    tids = pa.array(range(tid, tid + n), type=pa.int64())
    tid += n
    out = pa.record_batch(
        [b.column("cell"), tids, b.column("target"),
         b.column("x_global_px"), b.column("y_global_px")],
        schema=out_schema,
    )
    writer.write_batch(out)
    nrows += n
    nbatch += 1
    if nbatch % 50 == 0:
        print(f"  {nrows:,} kept rows, {time.time()-t:.0f}s", flush=True)
writer.close()
print(f"DONE tx->transcripts: {nrows:,} rows in {time.time()-t:.0f}s -> {OUT}")


## Step 4 — drive the Celldega Xenium pipeline → DegaFiles (non-row-group)
Reuses `celldega.pre.*` (technology=`Xenium`): cell metadata, leiden clustering, meta-gene, per-gene CBG parquets, cluster files, **vertically-flipped** morphology image pyramids from the Napari OME-ZARR (DNA + PanCK/Membrane/CD45), **streaming all-transcript** tiles (bounded memory, no subsample — celldega's `make_trx_tiles` would OOM loading 1.12B rows), cell boundary tiles, and `landscape_parameters.json` (classic non-row-group layout; row-groups have a known 403 issue when hosted on the Manifold bucket).

Coordinate notes: transform = `diag(0.25, 0.25, 1)` maps CosMx `global_px` → zarr level-2 px (`image_scale` stays 1 since celldega applies it inconsistently across functions); the image is `np.flipud`-ed to match the molecular layers.

In [ ]:
"""CosMx -> Celldega DegaFiles (non-row-group), driving the existing Xenium celldega.pre functions.

Self-contained. Assumes the Xenium-format intermediates already exist (see the convert_* steps):
  DATA_DIR/cells.csv.gz, cell_boundaries.parquet, cell_feature_matrix/ (10x MTX),
  transcripts.parquet (ALL transcripts, cols: cell_id, transcript_id, feature_name, x_location, y_location)

Produces classic (non-row-group) DegaFiles in DEGA. Key CosMx specifics handled here:
  * transform = diag(0.25,0.25,1): global_px (zarr level 0) -> level-2 image px. image_scale stays 1
    (celldega applies image_scale inconsistently across funcs; the matrix is applied uniformly).
  * morphology images read from the Napari OME-ZARR and vertically FLIPPED (np.flipud) to match the
    CosMx global_px frame used by cells/transcripts.
  * ALL ~1.12B transcripts are tiled via a streaming tiler (bounded memory) instead of celldega's
    make_trx_tiles, which would load the whole parquet into RAM (OOM). No subsampling.
  * gene-name reconciliation: exprMat uses '-' (e.g. C4B-2) but the tx file uses '_' (C4B_2).
"""
import re, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import pyvips
import zarr
import anndata as ad
import scipy.sparse as sp
import scanpy as sc
import celldega as dega
from celldega.pre.boundary_tile import _get_name_mapping

TECH = "Xenium"
DATA_DIR = Path("/home/jovyan/local/cosmx_xenium/S0")
DEGA = Path("/home/jovyan/local/cosmx_dega_files/S0_non-row-group")
DEGA.mkdir(parents=True, exist_ok=True)
TILE_SIZE = 250
COORD_SCALE = 0.25          # global_px -> zarr level 2 px
ZARR_LEVEL = "2"
MAX_WORKERS = 8
ZIP = "/home/jovyan/local/cosmx_data/Napari.zip"
ZPFX = "Run_b8806732-4c8c-4a36-bb0f-52a601a12475/20240620_231957_S2/Napari/images"
CHANNELS = [("DNA", "dapi", "DNA", [0, 0, 255]), ("PanCK", "panck", "PanCK", [0, 255, 0]),
            ("Membrane", "membrane", "Membrane", [255, 0, 255]), ("CD45", "cd45", "CD45", [255, 255, 0])]
IMAGE_INFO = [{"name": n, "button_name": b, "color": c} for (_, n, b, c) in CHANNELS]


def step(m):
    print(f"\n{'='*12} {m} {'='*12}", flush=True)


# 1. transform (0.25 scale, no flip of coords)
step("transform")
tpath = DEGA / "micron_to_image_transform.csv"
pd.DataFrame(np.diag([COORD_SCALE, COORD_SCALE, 1.0])).to_csv(tpath, sep=" ", header=False, index=False)

# 2. cell metadata (image px)
step("make_meta_cell_image_coord")
dega.pre.make_meta_cell_image_coord(TECH, str(tpath), str(DATA_DIR / "cells.csv.gz"),
                                    str(DEGA / "cell_metadata.parquet"), image_scale=1)

# 3. CBG (cache to h5ad: the 550M-nnz WTx MTX is slow to read from text)
step("read CBG")
h5 = DATA_DIR / "cbg.h5ad"
if h5.exists():
    adata = ad.read_h5ad(h5)
else:
    cbg0 = dega.pre.read_cbg_mtx(str(DATA_DIR / "cell_feature_matrix"), technology=TECH)
    adata = ad.AnnData(sp.csr_matrix(cbg0.sparse.to_coo()).astype(np.float32),
                       obs=pd.DataFrame(index=pd.Index(np.asarray(cbg0.index, dtype=str))),
                       var=pd.DataFrame(index=pd.Index(np.asarray(cbg0.columns, dtype=str))))
    adata.obs.index.name = None; adata.var.index.name = None
    adata.write_h5ad(h5); del cbg0
cbg = pd.DataFrame.sparse.from_spmatrix(adata.X, index=adata.obs_names, columns=adata.var_names)
assert str(cbg.index[0]).startswith("c_")

# 4. clustering -> clusters.csv (Xenium path expects analysis/clustering/.../clusters.csv)
step("clustering")
clusters_csv = DATA_DIR / "analysis" / "clustering" / "gene_expression_graphclust" / "clusters.csv"
if not clusters_csv.exists():
    cl = adata.copy()
    sc.pp.filter_cells(cl, min_counts=20); sc.pp.normalize_total(cl); sc.pp.log1p(cl)
    sc.pp.highly_variable_genes(cl, n_top_genes=2000); cl = cl[:, cl.var.highly_variable].copy()
    sc.pp.scale(cl, max_value=10); sc.tl.pca(cl, n_comps=50); sc.pp.neighbors(cl, n_neighbors=15, n_pcs=50)
    sc.tl.leiden(cl, resolution=1.0, flavor="igraph", n_iterations=2, directed=False)
    lab = (cl.obs["leiden"].astype(int) + 1).astype(str)
    df = pd.DataFrame({"Barcode": lab.index, "Cluster": lab.values})
    allc = pd.read_csv(DATA_DIR / "cells.csv.gz", usecols=["cell_id"])["cell_id"].astype(str)
    miss = set(allc) - set(df["Barcode"])
    if miss:
        df = pd.concat([df, pd.DataFrame({"Barcode": list(miss), "Cluster": "1"})], ignore_index=True)
    clusters_csv.parent.mkdir(parents=True, exist_ok=True); df.to_csv(clusters_csv, index=False)

# 5. meta gene, df_sig, cbg parquets (one per gene), clusters
step("meta_gene / df_sig / cbg / clusters")
dega.pre.make_meta_gene(cbg, str(DEGA / "meta_gene.parquet"))
dega.pre.cluster_gene_expression(TECH, str(DEGA), cbg, str(DATA_DIR))
dega.pre.save_cbg_gene_parquets(TECH, str(DEGA), cbg, verbose=True)
dega.pre.create_cluster_and_meta_cluster(TECH, str(DEGA), str(DATA_DIR))
del cbg, adata
import gc; gc.collect()

# 6. image deepzoom pyramids from the Napari zarr, ALIGNED to the flat-file frame, NOT packed.
#    The CosMx flat-file global-y and the napari image row are exact mirrors (row = C - y_global_px),
#    with the mirror axis C = H_img - fov_height (NOT H_img). So: np.flipud, then shift UP by ~one
#    fov_height. X already matches. We recover the exact (dy, dx) by registering the cell-density
#    raster against the flipped image (sample-agnostic). See COORDINATE_ALIGNMENT.md.
step("image pyramids (flipped + aligned)")
from skimage.registration import phase_cross_correlation
pyr = DEGA / "pyramid_images"; pyr.mkdir(exist_ok=True)
store = zarr.ZipStore(ZIP, mode="r"); g = zarr.open_group(store=store, path=ZPFX, mode="r")
# measure shift at level 5 (fast) -> scale to the working level
d5 = g["DNA"]["5"][:]; H5, W5 = d5.shape
_cm = pd.read_parquet(DEGA / "cell_metadata.parquet")
_xy = np.array(_cm["geometry"].to_list()) / (4.0 * (2 ** (5 - int(ZARR_LEVEL))))  # cell px -> level5 px
_cell = np.zeros((H5, W5), np.float32)
np.add.at(_cell, (np.clip(_xy[:, 1].astype(int), 0, H5 - 1), np.clip(_xy[:, 0].astype(int), 0, W5 - 1)), 1.0)
_sh, _, _ = phase_cross_correlation((_cell > 0).astype(np.float32),
                                    (d5[::-1] > d5.mean()).astype(np.float32), upsample_factor=4)
from scipy.ndimage import shift as _ndshift
_f = 2 ** (5 - int(ZARR_LEVEL))
DY, DX = _sh[0] * _f, _sh[1] * _f          # shift to APPLY to the flipped image at the working level
print(f"  image align shift (level {ZARR_LEVEL}): dy={DY:.1f} dx={DX:.1f}  (~one fov_height)")
# self-check the direction (phase_cross_correlation(ref=cells, moving=image) -> shift for image)
_chk = _ndshift((d5[::-1]).astype(np.float32), (_sh[0], _sh[1]), order=0, mode="constant", cval=0)
_res, _, _ = phase_cross_correlation((_cell > 0).astype(np.float32),
                                     (_chk > _chk.mean()).astype(np.float32), upsample_factor=4)
assert abs(_res[0]) < 3 and abs(_res[1]) < 3, f"image shift direction wrong (residual {_res})"

for cosmx_ch, name, _b, _c in CHANNELS:
    for p in (pyr / f"{name}_files",):
        if p.exists():
            shutil.rmtree(p)
    if (pyr / f"{name}.dzi").exists():
        (pyr / f"{name}.dzi").unlink()
    arr = _ndshift(np.flipud(g[cosmx_ch][ZARR_LEVEL][:]).copy(), (DY, DX),
                   order=0, mode="constant", cval=0)   # flip + align to flat-file frame
    samp = arr[::4, ::4].ravel(); samp = samp[samp > 0]
    lo, hi = (np.percentile(samp, (1, 99.5)) if samp.size else (0, 1)); del samp
    hi = float(max(hi, lo + 1)); lo = float(lo)
    np.clip(arr, lo, hi, out=arr); arr -= np.uint16(lo)
    arr8 = np.ascontiguousarray((arr * (255.0 / (hi - lo))).astype(np.uint8)); del arr
    h, w = arr8.shape
    pyvips.Image.new_from_memory(arr8.data, w, h, 1, "uchar").dzsave(
        str(pyr / name), tile_size=512, overlap=0, suffix=".webp[Q=90]")
    del arr8
    print(f"  {name} flipped {w}x{h}", flush=True)
store.close()

# 7. transcript tiles -- STREAM all transcripts (bounded memory), classic per-tile parquet layout
step("transcript tiles (streaming, ALL transcripts)")
gene_to_int = _get_name_mapping(str(DEGA), layer="transcript")
canon = lambda s: re.sub(r"[-_.]", "", str(s)).upper()
canon_to_name = {}
for gg in gene_to_int:
    canon_to_name.setdefault(canon(gg), gg)
def resolve(gene):
    if gene in gene_to_int:
        return gene_to_int[gene]
    nm = canon_to_name.get(canon(gene)); return gene_to_int[nm] if nm else None
src = DATA_DIR / "transcripts.parquet"
mx, my = pl.scan_parquet(src).select((pl.col("x_location").max() * COORD_SCALE),
                                     (pl.col("y_location").max() * COORD_SCALE)).collect().row(0)
nx, ny = int(np.ceil(mx / TILE_SIZE)) + 1, int(np.ceil(my / TILE_SIZE)) + 1
spill = DATA_DIR / "_trx_spill"
if spill.exists():
    shutil.rmtree(spill)
spill.mkdir()
seen = set()
for bi, batch in enumerate(pq.ParquetFile(str(src)).iter_batches(
        batch_size=8_000_000, columns=["feature_name", "x_location", "y_location"])):
    df = pl.from_arrow(batch).with_columns(
        pl.col("feature_name").replace_strict(
            {gv: resolve(gv) for gv in pl.from_arrow(batch)["feature_name"].unique().to_list()},
            default=None, return_dtype=pl.Int64).alias("name")).drop_nulls("name")
    df = df.with_columns((pl.col("x_location") * COORD_SCALE).round(2).alias("gx"),
                         (pl.col("y_location") * COORD_SCALE).round(2).alias("gy"))
    df = df.with_columns((pl.col("gx") / TILE_SIZE).floor().cast(pl.Int32).clip(0, nx - 1).alias("tx"),
                         (pl.col("gy") / TILE_SIZE).floor().cast(pl.Int32).clip(0, ny - 1).alias("ty")
                         ).with_columns(pl.concat_list(["gx", "gy"]).alias("geometry"))
    for (tx, ty), td in df.partition_by(["tx", "ty"], as_dict=True).items():
        d = spill / f"{tx}_{ty}"; d.mkdir(exist_ok=True)
        td.select(["name", "geometry"]).write_parquet(d / f"part_{bi:04d}.parquet")
        seen.add((tx, ty))
out = DEGA / "transcript_tiles"
if out.exists():
    shutil.rmtree(out)
out.mkdir()
for tx, ty in sorted(seen):
    parts = sorted((spill / f"{tx}_{ty}").glob("part_*.parquet"))
    pl.concat([pl.read_parquet(p) for p in parts]).to_pandas().to_parquet(
        out / f"transcripts_tile_{tx}_{ty}.parquet", index=False)
shutil.rmtree(spill)
print(f"  {len(seen)} transcript tiles")

# 8. cell boundary tiles
step("make_cell_boundary_tiles")
tile_bounds = {"x_min": 0.0, "y_min": 0.0, "x_max": float(mx), "y_max": float(my)}
dega.pre.make_cell_boundary_tiles(
    TECH, str(DATA_DIR / "cell_boundaries.parquet"), str(DEGA / "cell_segmentation"),
    str(DATA_DIR / "cells.csv.gz"), str(tpath), coarse_tile_factor=10, tile_size=TILE_SIZE,
    tile_bounds=tile_bounds, image_scale=1, max_workers=MAX_WORKERS)

# 9. landscape parameters (non-row-group)
step("save_landscape_parameters")
dega.pre.save_landscape_parameters(TECH, str(DEGA), image_name="dapi_files", tile_size=TILE_SIZE,
                                   image_info=IMAGE_INFO, image_format=".webp", use_int_index=True,
                                   use_row_groups=False)
print("\nDONE non-row-group CosMx DegaFiles at", DEGA)


## Step 5 — copy DegaFiles to the workbench (optional)
DegaFiles were written to fast local disk. Copy to the shared (s3fs) workbench when verified (prefer `aws s3 sync` directly to the bucket for the many small files).

In [ ]:
# import shutil
# shutil.copytree('/home/jovyan/local/cosmx_dega_files/S0_non-row-group',
#                 '/home/jovyan/workbench/CosMx_Celldega/cosmx_dega_files/S0_non-row-group')
print('DegaFiles: /home/jovyan/local/cosmx_dega_files/S0_non-row-group')